# Make cutouts for blinking

Given a pair of plates, this script creates two cutouts centered on a position specified by a source ID in a table.

The cutouts are writen as two FITS files, for ingestion and blinking by a FITS viewer app such as QFitsView or SAOimage.

In [1]:
import os
import json
from importlib import reload

import numpy as np

from astropy.io import fits
from astropy import units as u
from astropy.table import Table
from astropy.coordinates import SkyCoord
from astropy.wcs import WCS

import settings
from library import get_cutouts, update_dataset
from settings import RESULTS, get_parameters, current_dataset, fname

In [2]:
table = Table.read(os.path.join(RESULTS, "candidates_best_GS.fits"), format='fits')

In [3]:
# isolate data for this source in table
source_id = 40649930015591

mask = table['source_id'] == source_id
table = table[mask]

In [ ]:
# find the two images based on table entries
plate_id = str(table['plate_id_1'][0])
plate_id_next = str(table['plate_id_next'][0])

In [4]:
key = plate_id + ',' + plate_id_next
update_dataset(key)
reload(settings)
from settings import current_dataset

par = get_parameters(current_dataset)

image_name = fname(par['image1'])
image_name_next = fname(par['image2'])

In [5]:
# find coordinates to center cutouts

ra = table['ra_icrs'][0] * u.deg
dec = table['dec_icrs'][0] * u.deg

sky_coords = SkyCoord(ra=ra, dec=dec, frame='icrs')

In [6]:
# # alternative: we have already X,Y pixel coordinates - just convert them to RA,Dec
# # Using this method, make sure source ID is read from the table associated with the
# # correct images.
# x =
# y =

# # IMPORTANT: use the appropriate WCS. Here, we are using image2 as source of coordinates
# header = fits.getheader(image_name_next)
# wcs = WCS(header)

# world_coords = wcs.pixel_to_world(x, y)

# ra = np.array(world_coords.ra) * u.deg
# dec = np.array(world_coords.dec) * u.deg

# sky_coords = SkyCoord(ra=ra, dec=dec, frame='icrs')

In [7]:
# make the cutouts

cutout1, cutout2 = get_cutouts(image_name, image_name_next, sky_coords, 5. * u.deg)

Set MJD-AVG to 33709.060428 from DATE-AVG.
Set MJD-END to 33709.070810 from DATE-END'. [astropy.wcs.wcs]
Set MJD-AVG to 33709.088125 from DATE-AVG.
Set MJD-END to 33709.098519 from DATE-END'. [astropy.wcs.wcs]


In [8]:
# write cutouts to file

cutout_header_1 = cutout1.wcs.to_header()
cutout_header_2 = cutout2.wcs.to_header()

primary_hdu = fits.PrimaryHDU(data=cutout1.data, header=cutout_header_1)
image_hdu = fits.ImageHDU(data=cutout2.data, header=cutout_header_2)

primary_hdu.writeto('cutout1.fits', overwrite=True)
image_hdu.writeto('cutout2.fits', overwrite=True)

# hdul = fits.HDUList([primary_hdu, image_hdu])
# hdul.writeto('cutout_list.fits', overwrite=True)

In [9]:
par

{'nproc': 4,
 'nproc_analysis': 4,
 'use_catalog': True,
 'sextractor_flags': 8,
 'model_prediction': 0.5,
 'elongation': 2.5,
 'annular_bin': 5,
 'flag_rim': 0,
 'max_flux_threshold': 0.1,
 'fwhm_init': 10.0,
 'fit_shape': 33,
 'min_acceptable_flux': 500,
 'min_fwhm': 3.5,
 'max_fwhm': 20.0,
 'qfit_max': 2.5,
 'cfit_max': 0.003,
 'max_fit_flag': 8,
 'neighborhood_cutout_size': 15.0,
 'elongation_limit': 1.8,
 'profile_diff_threshold': 0.08,
 'circularity_threshold': [70],
 'circularity_low_limit': 0.8,
 'tiny_cutout_size': 25,
 'false_positive_threshold': 10.0,
 'fwhm_lim': 30.0,
 'disp_elong_lim': 1.5,
 'display_cutout_size': 2.5,
 'plot_limit': 200,
 'rotate': [False, False],
 'invert_east': [False, False],
 'invert_north': [False, False],
 'table1': 'sources_63059.csv',
 'table2': 'sources_63060.csv',
 'table1_calib': 'sources_calib_63059.csv',
 'table2_calib': 'sources_calib_63060.csv',
 'table_matched': 'table_match_63059_63060.fits',
 'table_psf_matched': 'table_psf_match_63059_